## GPT prompting: n80 examples, first filter

### requires python >= 3.10

In [1]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [2]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [3]:
from prompts.semantic_categories.v01.prompt import SYSTEM_PROMPT, FEW_SHOTS

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [5]:
RESULTS_DIR = "../../results/"

# algselt on 10K lauset võetud siit (esimesed 10K):
# "../../data/n80_examples_large_v01.csv"

# suurema andmestiku peal jookustamiseks on vaja v02 faili 
# ja jätta välja esimesed 10K (kui neid ei taha uuesti jooksutada)
DATA_FILE =  "../../data/n80_examples_large_v02.csv"

# muuda lõpus failinime: b10= batch 10 lause kaupa; run01 = esimest korda need andmed ette anda gpt-le
# gpt vastustega fail
GPT_ANSWER_FILE = RESULTS_DIR + "n80_examples_large_v02/gpt_v01/" + "gpt_b10_run01.csv"

CONF_FILE = 'azure.ini'


# OSA I : Andmed


## võtta andmefailist näited

In [6]:
df = pd.read_csv(DATA_FILE, encoding="utf-8",  sep=",")

In [7]:
## valida üks meetod: kas välja jätta juba testitud 10K näidet või võtta kõik
# kommenteeri välja see variant, mida ei kasuta

# 1) jätta välja esimesed 10K
#spatial_obl_ex = df.iloc[10000:]
#spatial_obl_ex = spatial_obl_ex.sample(frac=1)

# 2) kui ei jäta välja siis see:
# + shuffle (frac)
# enne/pärast sample ka .iloc[x:n] kui vaja võtta subset
spatial_obl_ex = df.sample(frac=1)

# testimiseks, võtta ainult 30 näidet, et workflow läbi teha 
# selleks võtta nt n80_examples_large_v01/gpt_v02 failist näiteid
#spatial_obl_ex = pd.concat([
#    df[df['classification2'] == 'no'].head(15),
#    df[df['classification2'] == 'yes'].head(15)])

#spatial_obl_ex = spatial_obl_ex[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
#       'morph_case', 'lemma', 'form', 'sentence', 'timex_tag','ekilex_tag', 'ner_tag']]

In [8]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
6,9344322,15008647,14,tervitama,NaN,in,ekstaas,ekstaasis,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .",NaN,NaN,NaN
8,11102573,17784543,6,ravima,NaN,in,seis,seisus,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",NaN,NaN,NaN
14,16252383,25172405,39,minema,ära,el,koosseis,koosseisust,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",NaN,NaN,NaN
27,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN
32,18375902,27858061,22,ütlema,NaN,ill,eelnev,eelnevasse,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,NaN,NaN,NaN
36,8099731,12997190,8,pöörama,NaN,ill,langus,langusesse,"Pigem on see madal või pöörab sootuks langusesse . """,NaN,NaN,NaN
37,10060758,16137721,6,jõudma,tagasi,el,lisaraha,lisarahast,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .",NaN,NaN,NaN
54,12633679,20215816,2,toimuma,NaN,ill,krae,kraesse,Raudteelaste kraesse toimunud õnnetusi Rentiku sõnul siiski veeretada ei saa .,NaN,NaN,NaN
59,9787399,15703680,13,maanduma,NaN,all,ema,emale,"Mõneski kultuuris tähendab öösel nähtud liblikas surma , või kui äsja sünnitanud emale maandub ööliblikas , siis sureb laps .",NaN,NaN,NaN
73,2891090,4635435,3,saama,NaN,adit,Risbiter,Risbiteri,[ Saab Risbiteri mõõga kätte .,NaN,NaN,NaN


# OSA II : GPT

## GPT jaoks vajalik

In [9]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [10]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

In [25]:
#SYSTEM_PROMPT

In [24]:
#FEW_SHOTS

## Andmete söötmine

### vajalikud funktsioonid

In [15]:
def classify_batch(my_batch):

    #attempt on juhuks kui ei saa tagasi õiget arvu vastuseid (juhtub 10-st suurema batchi puhul)
    # samas kui juba kord selle batchiga probleem, siis teist korda on ka probleem
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "Instruction": (
                "Analyse the few-shot examples. "
                "Then process the list called 'batch'. "
                "Output a JSON array with one item per batch entry, in the same order."
                "Output a JSON array of EXACTLY N items (same length as 'batch' list) in the same order. Do not add or remove items."
            ),
            "few_shots": FEW_SHOTS,
            "batch": json.dumps(my_batch, ensure_ascii=False)
        }
    
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False) }
        ]

        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [16]:
def explain_non_locations(
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    yes_subset_ratio: float = 0.0
) -> Dict[int, str]:

    # Determine which examples to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * yes_subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as location ('yes') or not location ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, with no markdown, no code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" + json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # Pydantic validation
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices

### df ja anda andmed sisse

In [17]:
df = spatial_obl_ex

In [18]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
6,9344322,15008647,14,tervitama,NaN,in,ekstaas,ekstaasis,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .",NaN,NaN,NaN
8,11102573,17784543,6,ravima,NaN,in,seis,seisus,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",NaN,NaN,NaN
14,16252383,25172405,39,minema,ära,el,koosseis,koosseisust,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",NaN,NaN,NaN
27,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN
32,18375902,27858061,22,ütlema,NaN,ill,eelnev,eelnevasse,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,NaN,NaN,NaN
36,8099731,12997190,8,pöörama,NaN,ill,langus,langusesse,"Pigem on see madal või pöörab sootuks langusesse . """,NaN,NaN,NaN
37,10060758,16137721,6,jõudma,tagasi,el,lisaraha,lisarahast,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .",NaN,NaN,NaN
54,12633679,20215816,2,toimuma,NaN,ill,krae,kraesse,Raudteelaste kraesse toimunud õnnetusi Rentiku sõnul siiski veeretada ei saa .,NaN,NaN,NaN
59,9787399,15703680,13,maanduma,NaN,all,ema,emale,"Mõneski kultuuris tähendab öösel nähtud liblikas surma , või kui äsja sünnitanud emale maandub ööliblikas , siis sureb laps .",NaN,NaN,NaN
73,2891090,4635435,3,saama,NaN,adit,Risbiter,Risbiteri,[ Saab Risbiteri mõõga kätte .,NaN,NaN,NaN


## NB! muuda max_allowed_tok kui vaja

In [19]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda, nt batch 10 lause kaupa bs = 10
bs = 10

# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 30000 #3200000


batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False) )

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    # kui ei ole vaja piirata raha/tokeite kulutust siis järgnevad read välja kommenteerida
    if used_tokens >= max_allowed_tok :
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

3it [00:08,  2.83s/it]


In [20]:
used_tokens # 30 lauset, batch 10-> 9954 tokenit, eelduslikult 10K lauset -> ~ 8 eur

9882

In [21]:
len(results)

30

## andmed tabelisse 

### enne kontroll kas andmeid on puudu ja vastavad lüngad täita

In [22]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

# pole probleemi
if len(results) == len(df):
    df["classification"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification"] = new_results
    df["explanation"] = new_explanations


In [23]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
6,9344322,15008647,14,tervitama,NaN,in,ekstaas,ekstaasis,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .",NaN,NaN,NaN,no,The word 'ekstaasis' describes an emotional state and not a specific location.
8,11102573,17784543,6,ravima,NaN,in,seis,seisus,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",NaN,NaN,NaN,no,"The word 'seisus' refers to a situation or condition, not a physical location."
14,16252383,25172405,39,minema,ära,el,koosseis,koosseisust,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",NaN,NaN,NaN,no,"The word 'koosseisust' refers to the composition or framework of an organization, not to a physical location."
27,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN,no,"The word 'hooldeprojekti' describes a plan or initiative, not a specific place."
32,18375902,27858061,22,ütlema,NaN,ill,eelnev,eelnevasse,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,NaN,NaN,NaN,no,"The word 'eelnevasse' refers to something previous or prior, and not a physical location."
36,8099731,12997190,8,pöörama,NaN,ill,langus,langusesse,"Pigem on see madal või pöörab sootuks langusesse . """,NaN,NaN,NaN,no,"The word 'langusesse' refers to a state or trend, not a physical or geographical place."
37,10060758,16137721,6,jõudma,tagasi,el,lisaraha,lisarahast,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .",NaN,NaN,NaN,no,"The word 'lisarahast' refers to additional funding or money, not a location."
54,12633679,20215816,2,toimuma,NaN,ill,krae,kraesse,Raudteelaste kraesse toimunud õnnetusi Rentiku sõnul siiski veeretada ei saa .,NaN,NaN,NaN,no,"The word 'kraesse' refers to accountability or blame, not a physical place."
59,9787399,15703680,13,maanduma,NaN,all,ema,emale,"Mõneski kultuuris tähendab öösel nähtud liblikas surma , või kui äsja sünnitanud emale maandub ööliblikas , siis sureb laps .",NaN,NaN,NaN,no,The word 'emale' refers to a person (mother) and not to a geographical or physical location.
73,2891090,4635435,3,saama,NaN,adit,Risbiter,Risbiteri,[ Saab Risbiteri mõõga kätte .,NaN,NaN,NaN,no,"The word 'Risbiteri' refers to a person or character, not a location."


### salvestada tulemused faili

In [52]:
df.to_csv(GPT_ANSWER_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

##### valikuline salvestamine, juhul kui andmeid oli puudu vms

fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)